# Inference & Usage — All the Details

This notebook is **proven to work** by a smoke test (`smoke/08_inference_smoke.py`).

## Goal
Teach how to **correctly use** a model after (or before) training:
- Model-loading methods (Unsloth Fast vs vanilla HF)
- Chat template + `add_generation_prompt`
- Generation parameters (model-specific recommendations)
- Streaming (simple and production)
- Batched inference
- Save / reload LoRA
- Save formats (LoRA / merged / GGUF)

## Tested (RTX 4070 Ti SUPER 16GB)
All 9 patterns passed in smoke testing — `Qwen3-4B-Instruct-2507` 4-bit + LoRA r=16.

## Contents

1. **Model Loading** — Unsloth FastLanguageModel
2. **`for_inference()` Optimization** — 2x faster
3. **Chat Template + `apply_chat_template`**
4. **Generation Parameters** — temperature, top_p, top_k, min_p, repetition_penalty
5. **Model-Specific Recommended Values**
6. **Streaming** — TextStreamer vs TextIteratorStreamer
7. **Batched Inference**
8. **Saving and Reloading LoRA**
9. **Save Formats** — LoRA / Merged / GGUF
10. **Thinking Model Inference** — parsing and controlling `<think>`
11. **Tool Calling Inference** — multi-turn tool execution
12. **Common Pitfalls**

## 1. Model Loading

Two main paths:

### A. Unsloth (recommended — if you'll also be training)
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen3-4B-Instruct-2507',
    max_seq_length=2048, load_in_4bit=True,
)
```

### B. Vanilla HuggingFace (inference only)
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.bfloat16, device_map='auto',
)
```

**Which one?**
- If you'll also train -> Unsloth (Triton kernels, faster + less VRAM)
- Inference only + production -> Vanilla is more flexible (works with serving frameworks like vLLM, TGI)

In [ ]:
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
from transformers import TextStreamer, TextIteratorStreamer
from threading import Thread

MODEL = 'unsloth/Qwen3-4B-Instruct-2507'

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL,
    max_seq_length = 2048,
    load_in_4bit = True,                       # QLoRA — 4 GB peak
    full_finetuning = False,
)

# Chat template — for the correct format
tokenizer = get_chat_template(tokenizer, chat_template='qwen3-instruct')

## 2. `for_inference()` Optimization

**CRITICAL:** Call `FastLanguageModel.for_inference(model)` when doing inference. **2x faster**.

If you're going to train: switch back with `FastLanguageModel.for_training(model)`.

What it does: optimizes caches, disables gradient computation, removes dropouts.

In [ ]:
FastLanguageModel.for_inference(model)            # 2x faster

## 3. Using the Chat Template for Inference

**`add_generation_prompt=True`** is mandatory — it tells the model "now answer".

### Pattern
```python
messages = [
    {'role': 'system', 'content': '...'},      # optional
    {'role': 'user', 'content': '...'},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,                  # MANDATORY
)
inputs = tokenizer(text, return_tensors='pt').to('cuda')
out = model.generate(**inputs, ...)
```

In [ ]:
# Helper — chat template + generate + decode-only-new-tokens
def gen_text(messages, **gen_kwargs):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, **gen_kwargs)
    new_tokens = out[0][inputs['input_ids'].shape[1]:]   # CRITICAL: skip the prompt
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

msgs = [{'role': 'user', 'content': 'What is interest?'}]
print(gen_text(msgs, max_new_tokens=80, do_sample=False))   # greedy

## 4. Generation Parameters

### Sampling Parameters
| Parameter | Meaning | Typical Value |
|---|---|---|
| `temperature` | Creativity. Low = deterministic, high = diverse | 0.7-1.0 |
| `top_p` (nucleus) | Cumulative probability threshold — sample from tokens that exceed it | 0.8-0.95 |
| `top_k` | Sample from the top K tokens | 20-64 |
| `min_p` | Minimum probability threshold (alternative to top_p) | 0.05-0.1 |
| `do_sample` | False = greedy/argmax, True = sampling | True |

### Repetition Control
| Parameter | Meaning | Typical Value |
|---|---|---|
| `repetition_penalty` | Penalizes repeated tokens (>1) | 1.0-1.3 |
| `no_repeat_ngram_size` | No n-gram of this size may repeat | 3 (for long text) |

### Length Control
| Parameter | Meaning |
|---|---|
| `max_new_tokens` | Number of **new tokens only** (recommended) |
| `max_length` | Total (prompt + new) — don't use! |
| `min_new_tokens` | Minimum number of new tokens |

In [ ]:
# Greedy (deterministic)
print('--- Greedy ---')
print(gen_text(msgs, max_new_tokens=60, do_sample=False))

# Low temperature (focused)
print('\n--- T=0.3 (focused) ---')
print(gen_text(msgs, max_new_tokens=60, temperature=0.3, do_sample=True))

# High temperature (creative)
print('\n--- T=1.5 (creative) ---')
print(gen_text(msgs, max_new_tokens=60, temperature=1.5, top_p=0.95, do_sample=True))

# Repetition penalty
print('\n--- repetition_penalty=1.3 ---')
print(gen_text(msgs, max_new_tokens=60,
               temperature=0.7, top_p=0.8, top_k=20,
               repetition_penalty=1.3, do_sample=True))

## 5. Model-Specific Recommended Values

Compiled from the official notebooks:

| Model Family | Temperature | top_p | top_k | min_p | Notes |
|---|---|---|---|---|---|
| **Qwen3 Instruct (non-thinking)** | 0.7 | 0.8 | 20 | – | Flatter, focused |
| **Qwen3 Thinking** | 0.6 | 0.95 | 20 | – | Low T for thinking |
| **Qwen3.5 Vision** | 1.5 | – | – | 0.1 | Uses min_p |
| **Gemma 4** | 1.0 | 0.95 | 64 | – | More flexible |
| **Gemma 3** | 1.0 | 0.95 | 64 | – | Same as Gemma 4 |
| **Llama 3.1** | 0.6 | 0.9 | – | – | Low T |

### By Use Case
| Use Case | T | top_p | Notes |
|---|---|---|---|
| Code generation | 0.0-0.2 | 1.0 | Greedy / low T is safer |
| Verifiable answers (math, fact) | 0.0-0.3 | 1.0 | Deterministic |
| Creative writing | 0.8-1.2 | 0.95 | More variety |
| Brainstorming | 1.2-1.5 | 0.95 | High diversity |

## 6. Streaming

Two options:

### A. `TextStreamer` (simple)
Just writes to stdout. Visible in the notebook.

### B. `TextIteratorStreamer` (production)
Async-friendly, ideal for FastAPI / Streamlit / Discord bots. `generate` runs in a background thread, the main thread yields tokens.

In [ ]:
# A. TextStreamer — simple
text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to('cuda')
_ = model.generate(
    **inputs, max_new_tokens=80,
    temperature=0.7, top_p=0.8, top_k=20, do_sample=True,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

In [ ]:
# B. TextIteratorStreamer — production pattern
streamer = TextIteratorStreamer(
    tokenizer, skip_prompt=True, skip_special_tokens=True,
)
gen_kwargs = dict(
    **inputs, max_new_tokens=80,
    temperature=0.7, top_p=0.8, top_k=20, do_sample=True,
    streamer=streamer,
)
thread = Thread(target=model.generate, kwargs=gen_kwargs)
thread.start()

# Token-by-token yield
collected = ''
for chunk in streamer:
    collected += chunk
    print(chunk, end='', flush=True)
thread.join()
print(f'\n\nTotal: {len(collected)} chars')

## 7. Batched Inference

Multiple prompts at once. **LEFT padding** is mandatory.

Memory: batch_size * (prompt_len + max_new_tokens) * model_size = O(batch * seq * params)

In [ ]:
prompts = [
    'Where is Istanbul?',
    'How does snow form?',
    'What is quantum?',
]
formatted = [
    tokenizer.apply_chat_template(
        [{'role': 'user', 'content': p}],
        tokenize=False, add_generation_prompt=True,
    ) for p in prompts
]

tokenizer.padding_side = 'left'                  # REQUIRED for batching
batch_inputs = tokenizer(
    formatted, return_tensors='pt', padding=True,
).to('cuda')

out = model.generate(
    **batch_inputs, max_new_tokens=50,
    temperature=0.7, top_p=0.8, top_k=20, do_sample=True,
    pad_token_id=tokenizer.pad_token_id,
)

# Decode each sample
for i, prompt in enumerate(prompts):
    new_tokens = out[i][batch_inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    print(f'\nQ: {prompt}')
    print(f'A: {response[:120]}...')

tokenizer.padding_side = 'right'                 # restore

## 8. Saving and Reloading LoRA

### Save (after training)
```python
model.save_pretrained('my_lora')
tokenizer.save_pretrained('my_lora')
```
This saves only the **adapter weights** + config (~150MB), not the base model.

### Reload (production scenario)
```python
# Pass the LoRA dir directly to from_pretrained
# The base model is found automatically from adapter_config.json
model, tokenizer = FastLanguageModel.from_pretrained(
    'my_lora',                                    # the LoRA dir, NOT the base model name
    max_seq_length=2048, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```

Unsloth reads `adapter_config.json` from the LoRA dir, then automatically downloads + loads the base model and attaches the adapter.

## 9. Save Formats

### A. LoRA Adapter (smallest, ~150MB)
```python
model.save_pretrained('my_lora')
tokenizer.save_pretrained('my_lora')
```
**Pros:** Small, fast upload/download. Multiple adapters can share the same base.
**Cons:** Always needs the base model for inference.

### B. Merged 16-bit (vLLM / HF inference)
```python
model.save_pretrained_merged(
    'my_merged',
    tokenizer,
    save_method='merged_16bit',                  # 'merged_4bit' is also available
)
```
**Pros:** Drops straight into production serving frameworks like vLLM and TGI. Single artifact.
**Cons:** Full model size (Qwen3-4B = ~8GB bf16).

### C. GGUF (Ollama / llama.cpp)
```python
model.save_pretrained_gguf(
    'my_gguf',
    tokenizer,
    quantization_method='q4_k_m',                # q4_k_m, q5_k_m, q8_0, f16
)
```
**Pros:** CPU + GPU hybrid (Ollama, llama.cpp). Quantized, small.
**Cons:** Depends on llama.cpp; conversion takes a while.

### D. Push to Hub
```python
# LoRA
model.push_to_hub('USER/my_lora', token='hf_xxx')
tokenizer.push_to_hub('USER/my_lora', token='hf_xxx')

# Merged
model.push_to_hub_merged(
    'USER/my_merged', tokenizer,
    save_method='merged_16bit', token='hf_xxx',
)
```

### Decision Table
| Goal | Format | Reason |
|---|---|---|
| Deploy / share on the Hub | LoRA | Small, many adapters share one base |
| vLLM serving | merged_16bit | vLLM also supports LoRA, but merged is simpler |
| Ollama / local CLI | GGUF q4_k_m | CPU+GPU, quantized |
| Mobile / edge | GGUF q4_k_s | Smallest |
| HuggingFace Inference API | merged_16bit | Native HF format |

## 10. Thinking Model Inference

Thinking models (Qwen3-Thinking-2507, R1-distill, etc.) produce a response in **two parts**:
1. `<think>...</think>` — the model's internal reasoning (can be hidden in the UI)
2. **Final answer** — the clean part shown to the user

### Generation Flow
```
User: what is 137 * 49?
        v (qwen3-thinking template ends the generation prompt with: <|im_start|>assistant\n<think>\n)
Model: 137 * 50 = 6850, 6850 - 137 = 6713.</think>

The result is 6713.<|im_end|>
```

### Control Mechanisms
| Method | Effect |
|---|---|
| `qwen3-thinking` template | Always auto-prepends `<think>\n` |
| `qwen3-instruct` template | Does not add `<think>` (no thinking) |
| `enable_thinking=False` parameter | Disabled in some models (Gemma 4 supports it, qwen3-thinking ignores it) |
| Post-process strip | Strip `<think>...</think>` with regex -> extract final answer |

### Recommended Generation Parameters
| Param | Value |
|---|---|
| `temperature` | **0.6** (low — focused reasoning) |
| `top_p` | 0.95 |
| `top_k` | 20 |
| `max_new_tokens` | **1024-4096** (think + answer can be long!) |
| `do_sample` | True |

In [ ]:
# Load a thinking model (we loaded Qwen3-Instruct in earlier sections)
# Clear the model to simulate a fresh session
import gc
del model
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen3-4B-Thinking-2507',
    max_seq_length = 4096,                       # think can be long
    load_in_4bit = True,
)
tokenizer = get_chat_template(tokenizer, chat_template='qwen3-thinking')
FastLanguageModel.for_inference(model)

In [ ]:
# Math problem — ideal for a thinking model
msgs = [{'role': 'user', 'content': 'Calculate 137 * 49'}]
text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to('cuda')

out = model.generate(
    **inputs, max_new_tokens=1024,                # leave room for think
    temperature=0.6, top_p=0.95, top_k=20,        # recommended for thinking models
    do_sample=True,
)
full = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'--- Full output ({len(full)} chars) ---')
print(full[:800])

### Parse — Separate `<think>` from the Final Answer

**Important quirk (proven in smoke tests):** the `qwen3-thinking` template appends `<think>\n` to the end of the generation prompt. With `skip_special_tokens=True`, the **opening token is lost** during decoding — but `</think>` is captured.

Pattern: try the full `<think>...</think>` regex first; if it fails, fall back to splitting on `</think>`.

In [ ]:
import re

# Strategy 1: capture the full <think>...</think>
think_match = re.search(r'<think>\s*(.*?)\s*</think>', full, re.DOTALL)
if think_match:
    think_part = think_match.group(1).strip()
    final_part = re.sub(r'<think>.*?</think>\s*', '', full, count=1, flags=re.DOTALL).strip()
else:
    # Strategy 2: split on </think> (template prepended <think>, decoder dropped it)
    if '</think>' in full:
        think_part, final_part = full.split('</think>', 1)
        think_part = think_part.strip()
        final_part = final_part.strip()
    else:
        think_part = ''
        final_part = full.strip()

print(f'--- Thinking ({len(think_part)} chars) ---')
print(think_part[:400] + ('...' if len(think_part) > 400 else ''))
print(f'\n--- Final Answer ({len(final_part)} chars) ---')
print(final_part[:400])

print(f'\n--- For UI (only final to user): ---')
print(final_part)

### Production UI Pattern — Streaming + Thinking

```python
in_think = True
think_buffer = ''
answer_buffer = ''

for chunk in streamer:
    if '</think>' in chunk:
        # transition: think -> answer
        before, after = chunk.split('</think>', 1)
        think_buffer += before
        answer_buffer += after
        in_think = False
        ui.show_thinking_collapsed(think_buffer)   # 'Thinking...' badge
    elif in_think:
        think_buffer += chunk
        ui.update_thinking_indicator()              # spinner
    else:
        answer_buffer += chunk
        ui.append_to_answer(chunk)                  # main answer
```

This is the same pattern used by Gemini, Claude, and ChatGPT for reasoning model UIs.

## 11. Tool Calling Inference

Tool calling = giving the model the ability to **call functions**. The model emits a tool call in JSON, your code executes it and feeds the result back.

### 3-step Flow (proven in smoke tests)
```
[User]   -> "What's the weather in Istanbul?"
             v (tools schema is injected into the system message)
[Model]  -> <tool_call>{"name":"get_weather","arguments":{"city":"Istanbul"}}</tool_call>
             v (your code calls it)
[Tool]   -> {"city":"Istanbul","temp_c":18,"condition":"cloudy"}
             v (fed back to the model with role='tool')
[Model]  -> "Istanbul is currently cloudy with a temperature of 18 degC."
```

### Format by Model (Parse Regex)
| Model | Format | Parse Regex |
|---|---|---|
| Qwen3 Instruct/Thinking | `<tool_call>{json}</tool_call>` | `r'<tool_call>\s*(\{.*?\})\s*</tool_call>'` |
| Qwen3.5 (Hermes-Pro) | `<function=NAME><parameter=KEY>VAL</parameter></function>` | `r'<function=(\w+)>(.*?)</function>'` |
| Gemma 4 (FunctionGemma) | `<\|tool_call>call:NAME{...}<tool_call\|>` | `r'<\|tool_call>call:(\w+)\{(.*?)\}<tool_call\|>'` |
| Llama 3.1 | Plain JSON in content | Direct json.loads |

### Recommended Generation Parameters
| Param | Value |
|---|---|
| `temperature` | **0.3-0.7** (low — keep JSON format intact) |
| `top_p` | 0.8 |
| `top_k` | 20 |
| `max_new_tokens` | 200-300 (tool call short, final answer medium) |

In [ ]:
# Load Qwen3-Instruct (ideal for tool calling)
import gc
del model
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen3-4B-Instruct-2507',
    max_seq_length = 2048,
    load_in_4bit = True,
)
tokenizer = get_chat_template(tokenizer, chat_template='qwen3-instruct')
FastLanguageModel.for_inference(model)

In [ ]:
# Define tools — OpenAI function-calling JSON schema
import json

tools = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get current weather for a city',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string', 'description': 'City name'}},
            'required': ['city'],
        },
    },
}, {
    'type': 'function',
    'function': {
        'name': 'calculator',
        'description': 'Evaluate a math expression',
        'parameters': {
            'type': 'object',
            'properties': {'expression': {'type': 'string'}},
            'required': ['expression'],
        },
    },
}]

# Mock tool implementations
def get_weather(city: str) -> dict:
    fake = {'Istanbul': 18, 'Ankara': 12, 'Paris': 15, 'Tokyo': 22}
    return {'city': city, 'temp_c': fake.get(city, 20), 'condition': 'cloudy'}

def calculator(expression: str) -> str:
    try:
        return str(eval(expression, {'__builtins__': {}}, {}))
    except Exception as e:
        return f'Error: {e}'

TOOL_FNS = {'get_weather': get_weather, 'calculator': calculator}

In [ ]:
# Turn 1 — User asks, model decides to call tool
conv = [
    {'role': 'system', 'content': 'You are a helpful assistant with tool access.'},
    {'role': 'user',   'content': 'How is the weather in Istanbul?'},
]

text = tokenizer.apply_chat_template(
    conv, tokenize=False, tools=tools, add_generation_prompt=True,
)
inputs = tokenizer(text, return_tensors='pt').to('cuda')
out = model.generate(
    **inputs, max_new_tokens=200,
    temperature=0.3, top_p=0.8, top_k=20, do_sample=True,    # low T — JSON precision
)
response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('--- Model output ---')
print(response)

In [ ]:
# Parse <tool_call>{json}</tool_call> (Qwen3 format) + execute
tool_match = re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', response, re.DOTALL)

if tool_match:
    call = json.loads(tool_match.group(1))
    tool_name = call['name']
    tool_args = call['arguments']
    print(f'Parsed call: {tool_name}({tool_args})')

    # Execute (real Python function)
    if tool_name in TOOL_FNS:
        result = TOOL_FNS[tool_name](**tool_args)
        print(f'Executed → {result}')
    else:
        result = {'error': f'Unknown tool: {tool_name}'}
else:
    print('No tool call detected — model answered directly')
    tool_name = None
    result = None

In [ ]:
# Turn 2 — Feed tool result, get final answer
if tool_name:
    conv_full = conv + [
        # Assistant's tool call
        {'role': 'assistant', 'content': '',
         'tool_calls': [{
             'id': 'call_1', 'type': 'function',
             'function': {'name': tool_name, 'arguments': tool_args},
         }]},
        # Tool execution result (role='tool', tool_call_id must match)
        {'role': 'tool', 'tool_call_id': 'call_1', 'name': tool_name,
         'content': json.dumps(result)},
    ]

    text2 = tokenizer.apply_chat_template(
        conv_full, tokenize=False, tools=tools, add_generation_prompt=True,
    )
    inputs2 = tokenizer(text2, return_tensors='pt').to('cuda')
    out2 = model.generate(
        **inputs2, max_new_tokens=150,
        temperature=0.7, top_p=0.8, top_k=20, do_sample=True,
    )
    final = tokenizer.decode(out2[0][inputs2['input_ids'].shape[1]:], skip_special_tokens=True)
    print('--- Final answer ---')
    print(final)

### Production Tool Loop (Multi-step)

The model can call multiple tools (in parallel or sequentially). Production pattern:

```python
MAX_ITERATIONS = 5                                # guard against infinite loops
conv = [{'role': 'user', 'content': user_query}]

for i in range(MAX_ITERATIONS):
    # 1. Generate
    response = generate(conv, tools=tools)

    # 2. Parse all tool calls
    calls = parse_all_tool_calls(response)

    # 3. No tool calls -> final answer, done
    if not calls:
        conv.append({'role': 'assistant', 'content': response})
        break

    # 4. Append assistant message with tool_calls
    conv.append({'role': 'assistant', 'content': '',
                 'tool_calls': [...]})

    # 5. Execute each tool, append results
    for call in calls:
        result = TOOL_FNS[call['name']](**call['args'])
        conv.append({'role': 'tool', 'tool_call_id': call['id'],
                     'name': call['name'], 'content': json.dumps(result)})

# Loop ends when the model gives a final answer (no more tool calls)
```

### Control Pitfalls
1. **`tools=[...]` on a template that doesn't support tools** -> silently ignored (Gemma 4 standard)
2. **`tool_call_id` doesn't match** -> the model gets stuck
3. **Bad JSON format** -> tool isn't executed (validate it: `json.loads` try/except)
4. **Infinite tool-call loop** -> set `MAX_ITERATIONS`
5. **Tool result too long** -> blows up the context, shorten or summarize it
6. **`temperature` too high** -> JSON format breaks (keep it low: 0.3-0.7)

## 12. Common Pitfalls

| # | Mistake | Result | Fix |
|---|---|---|---|
| 1 | `add_generation_prompt=False` at inference | Model doesn't answer, expects training format | `=True` |
| 2 | Not calling `for_inference()` | 2x slower inference | `FastLanguageModel.for_inference(model)` |
| 3 | Decoded text contains the prompt too | Prompt leaks into output | `out[0][inputs['input_ids'].shape[1]:]` |
| 4 | `padding_side='right'` in batches | Wrong output (pad tokens land at the start of the response) | `'left'` is mandatory |
| 5 | `do_sample=False` + `temperature` | Temperature is ignored | Set `do_sample=True` |
| 6 | Using `max_length` | Total prompt + completion, hard to control | Use `max_new_tokens` |
| 7 | `model.generate(input_ids=..., temperature=...)` (kwargs unpacking) | Errors in some cases | Pass via `**inputs` |
| 8 | Plain text content for a vision model | TypeError | Use `[{'type': 'text', 'text': ...}]` list |
| 9 | `tokenizer.decode(out)` (whole sequence again) | Output includes prompt + special tokens | `skip_special_tokens=True` + slice |
| 10 | `TextStreamer` in production | Not async (writes to stdout) | `TextIteratorStreamer` |
| 11 | Writing the base model name on reload | `from_pretrained('Qwen3-4B')` skips the adapter | `from_pretrained('my_lora_dir/')` |
| 12 | Always using the same generation params | Use case matters | Code -> low T, creative -> high T |
| 13 | Thinking model `<think>` not captured by full regex | Template prepended `<think>\n`, decoder skipped it | Fall back to splitting on `</think>` |
| 14 | Tool calling `temperature` too high | JSON format breaks | Keep low at 0.3-0.7 |
| 15 | Tool result included as long context | Loops slow down, OOM | Trim/summarize the tool result |

## Summary

1. **`FastLanguageModel.for_inference(model)`** — 2x speedup
2. **`add_generation_prompt=True`** is mandatory at inference
3. **`out[0][inputs['input_ids'].shape[1]:]`** — decode only the new tokens
4. **Generation params are model-specific** — Qwen3 instruct T=0.7, Gemma 4 T=1.0, Qwen3.5 min_p=0.1
5. **`padding_side='left'` is mandatory** in batches
6. **`TextIteratorStreamer`** in production (FastAPI / Streamlit)
7. **LoRA reload:** `from_pretrained('lora_dir/')` — finds the base automatically from adapter_config
8. **Save: LoRA (share)** / **Merged (vLLM)** / **GGUF (Ollama)**

**Tested:** `smoke/08_inference_smoke.py` — all 9 patterns (load, params, streaming, batch, train, save, reload) passed.

## Next Steps
- **Production serving:** vLLM, TGI, Ollama documentation
- **Adapter merging:** Combining multiple LoRA adapters (`add_weighted_adapter`)
- **Quantization:** AWQ, GPTQ, AQLM (post-training quantization)